In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fake-and-real-news-dataset' dataset.
Path to dataset files: /kaggle/input/fake-and-real-news-dataset


In [2]:
import os
print(os.listdir(path))

['True.csv', 'Fake.csv']


In [3]:
import pandas as pd

fake_df = pd.read_csv("/kaggle/input/fake-and-real-news-dataset/Fake.csv")
true_df = pd.read_csv("/kaggle/input/fake-and-real-news-dataset/True.csv")

print(fake_df.shape)
print(true_df.shape)

fake_df.head()

(23481, 4)
(21417, 4)


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [4]:
fake_df["label"] = 0
true_df["label"] = 1

In [5]:
df = pd.concat([fake_df, true_df], axis=0)

print(df.shape)

df.sample(5)

(44898, 5)


,title,text,subject,date,label
20879,Exclusive: Bangladesh protests over Myanmar's ...,DHAKA (Reuters) - Bangladesh lodged a protest ...,worldnews,"September 6, 2017",1
6279,The Pope Just Called Most American Employers ...,Pope Francis is no fan of Walmart or of McDona...,News,"May 19, 2016",0
17437,BREAKING: WHY DID MASSACHUSETTS OFFICIALS WAIT...,Where was the media coverage when this black t...,Government News,"Apr 10, 2015",0
14759,WHO NEEDS DEMOCRATS? GOP Consultant Says Estab...,No wonder we ve had the worst Democrat Preside...,politics,"Dec 23, 2015",0
390,Trump Shamefully Uses Hurricane Devastation T...,Leave it to Donald Trump to exploit a tragedy....,News,"September 13, 2017",0


In [6]:
df = df[["text", "label"]]

df.head()

,text,label
0,Donald Trump just couldn t wish all Americans ...,0
1,House Intelligence Committee Chairman Devin Nu...,0
2,"On Friday, it was revealed that former Milwauk...",0
3,"On Christmas day, Donald Trump announced that ...",0
4,Pope Francis used his annual Christmas Day mes...,0


In [7]:
df = df.sample(frac=1, random_state=42)

df.reset_index(drop=True, inplace=True)

df.head()

,text,label
0,"21st Century Wire says Ben Stein, reputable pr...",0
1,WASHINGTON (Reuters) - U.S. President Donald T...,1
2,(Reuters) - Puerto Rico Governor Ricardo Rosse...,1
3,"On Monday, Donald Trump once again embarrassed...",0
4,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",1


In [8]:
df.isnull().sum()

,0
text,0
label,0


In [9]:
import re

def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"www\S+", "", text)

    text = re.sub(r"[^a-zA-Z ]", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

df["clean_text"] = df["text"].apply(clean_text)

In [10]:
from sklearn.model_selection import train_test_split

X = df["clean_text"]

y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(35918,)
(8980,)


In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer

MAX_WORDS = 10000

tokenizer = Tokenizer(num_words=MAX_WORDS)

tokenizer.fit_on_texts(X_train)

In [12]:
X_train_seq = tokenizer.texts_to_sequences(X_train)

X_test_seq = tokenizer.texts_to_sequences(X_test)

In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 300

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding='post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding='post'
)

print(X_train_pad.shape)
print(X_test_pad.shape)

(35918, 300)
(8980, 300)


In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Flatten

ann_model = Sequential([

    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),

    Flatten(),

    Dense(128, activation='relu'),

    Dense(1, activation='sigmoid')
])

ann_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [15]:
ann_history = ann_model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 66s 142ms/step - accuracy: 0.9740 - loss: 0.0658 - val_accuracy: 0.9923 - val_loss: 0.0225
Epoch 2/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 63s 140ms/step - accuracy: 0.9994 - loss: 0.0024 - val_accuracy: 0.9915 - val_loss: 0.0316
Epoch 3/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 55s 122ms/step - accuracy: 0.9999 - loss: 7.4863e-04 - val_accuracy: 0.9915 - val_loss: 0.0282
Epoch 4/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 55s 122ms/step - accuracy: 1.0000 - loss: 5.4809e-04 - val_accuracy: 0.9915 - val_loss: 0.0280
Epoch 5/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 56s 124ms/step - accuracy: 1.0000 - loss: 5.1131e-04 - val_accuracy: 0.9915 - val_loss: 0.0288


In [16]:
ann_loss, ann_acc = ann_model.evaluate(
    X_test_pad,
    y_test
)

print("ANN Accuracy:", ann_acc)

281/281 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.9931 - loss: 0.0238
ANN Accuracy: 0.9930957555770874


In [17]:
from tensorflow.keras.layers import SimpleRNN

rnn_model = Sequential([

    Embedding(MAX_WORDS, 128),

    SimpleRNN(64),

    Dense(1, activation='sigmoid')
])

rnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

rnn_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [18]:
rnn_history = rnn_model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 71s 153ms/step - accuracy: 0.7426 - loss: 0.4336 - val_accuracy: 0.7695 - val_loss: 0.3976
Epoch 2/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 66s 147ms/step - accuracy: 0.8076 - loss: 0.3508 - val_accuracy: 0.7358 - val_loss: 0.4095
Epoch 3/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 64s 142ms/step - accuracy: 0.7728 - loss: 0.3970 - val_accuracy: 0.7334 - val_loss: 0.4325
Epoch 4/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 84s 145ms/step - accuracy: 0.7714 - loss: 0.3611 - val_accuracy: 0.7479 - val_loss: 0.3813
Epoch 5/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 64s 142ms/step - accuracy: 0.7875 - loss: 0.3079 - val_accuracy: 0.7738 - val_loss: 0.3730


In [19]:
rnn_loss, rnn_acc = rnn_model.evaluate(
    X_test_pad,
    y_test
)

print("RNN Accuracy:", rnn_acc)

281/281 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.7636 - loss: 0.3821
RNN Accuracy: 0.7635857462882996


In [20]:
from tensorflow.keras.layers import LSTM

lstm_model = Sequential([

    Embedding(MAX_WORDS, 128),

    LSTM(64),

    Dense(1, activation='sigmoid')
])

lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [21]:
lstm_history = lstm_model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 182s 399ms/step - accuracy: 0.7594 - loss: 0.4509 - val_accuracy: 0.8047 - val_loss: 0.3521
Epoch 2/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 183s 409ms/step - accuracy: 0.9214 - loss: 0.2037 - val_accuracy: 0.9365 - val_loss: 0.1831
Epoch 3/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 177s 394ms/step - accuracy: 0.9463 - loss: 0.1584 - val_accuracy: 0.9492 - val_loss: 0.1512
Epoch 4/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 179s 397ms/step - accuracy: 0.9595 - loss: 0.1153 - val_accuracy: 0.9747 - val_loss: 0.0962
Epoch 5/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 222s 442ms/step - accuracy: 0.9866 - loss: 0.0510 - val_accuracy: 0.9840 - val_loss: 0.0612


In [22]:
lstm_loss, lstm_acc = lstm_model.evaluate(
    X_test_pad,
    y_test
)

print("LSTM Accuracy:", lstm_acc)

281/281 ━━━━━━━━━━━━━━━━━━━━ 15s 53ms/step - accuracy: 0.9854 - loss: 0.0585
LSTM Accuracy: 0.9854120016098022


In [23]:
results = pd.DataFrame({
    "Model": ["ANN", "SimpleRNN", "LSTM"],
    "Accuracy": [ann_acc, rnn_acc, lstm_acc]
})

print(results)

       Model  Accuracy
0        ANN  0.993096
1  SimpleRNN  0.763586
2       LSTM  0.985412


In [24]:
lstm_model.save("fake_news_lstm.h5")

In [25]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [26]:
def predict_news(news):

    news = clean_text(news)

    seq = tokenizer.texts_to_sequences([news])

    pad = pad_sequences(
        seq,
        maxlen=MAX_LEN,
        padding='post'
    )

    pred = lstm_model.predict(pad)

    if pred[0][0] > 0.5:
        return "Real News"
    else:
        return "Fake News"

In [27]:
sample = """
The government announced a new economic policy today.
"""

print(predict_news(sample))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 328ms/step
Fake News


**Why LSTM Handles Long Text Better?**

SimpleRNN suffers from the vanishing gradient problem. Information from earlier words gradually disappears.

LSTM introduces memory cells and gates:

Forget Gate
Input Gate
Output Gate

These allow important information to be retained across hundreds of words.

For a long news article:

ANN ignores word order.
SimpleRNN remembers only short-term context.
LSTM remembers long-term context.

Hence LSTM usually achieves the highest accuracy for fake news detection.